# 第六章：对话引擎与多文档 Agent

## 学习目标
- 构建带对话记忆的 ChatEngine
- 理解不同对话模式（condense_question / context / condense_plus_context）
- 将查询引擎包装为工具（QueryEngineTool）
- 使用 RouterQueryEngine 自动路由到不同知识库
- 对比 LangChain 的对话和 Agent 模式

## 0. 环境准备

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

from llama_index.llms.openai_like import OpenAILike
from llama_index.embeddings.openai_like import OpenAILikeEmbedding
from llama_index.core import Settings

# 初始化 LLM（通过 OpenAI 兼容接口接入 Qwen）
llm = OpenAILike(
    model="qwen-plus",
    api_base=os.environ["Qwen_API_BASE"],
    api_key=os.environ["Qwen_API_KEY"],
    is_chat_model=True,
)
Settings.llm = llm

# 切换为 GLM（同样是 OpenAI 兼容接口）：
# from llama_index.llms.openai_like import OpenAILike
# Settings.llm = OpenAILike(model="glm-4-plus", api_base=os.environ["GLM_API_BASE"], api_key=os.environ["GLM_API_KEY"], is_chat_model=True)

# 初始化 Embedding 模型
Settings.embed_model = OpenAILikeEmbedding(
    model_name="text-embedding-v3",
    api_base=os.environ["Qwen_API_BASE"],
    api_key=os.environ["Qwen_API_KEY"],
)

print("环境初始化完成")

In [ ]:
# 准备示例文档和索引（后续多个小节复用）
from llama_index.core import Document, VectorStoreIndex

rag_docs = [
    Document(text="RAG（Retrieval-Augmented Generation，检索增强生成）是一种结合检索和生成的技术。它先从知识库中检索相关文档，再将检索结果作为上下文交给 LLM 生成回答。RAG 可以有效减少 LLM 的幻觉问题，让回答更加准确和有据可依。"),
    Document(text="RAG 的核心步骤包括：1) 文档加载与切分；2) 向量化存储（Embedding）；3) 用户提问时检索相关文档；4) 将检索结果和问题一起发给 LLM 生成回答。这四个步骤构成了一个完整的 RAG 管线。"),
    Document(text="RAG 的优势包括：不需要微调模型就能利用私有数据；知识库可以随时更新而不需要重新训练；可以提供信息来源的引用；成本远低于微调大模型。RAG 已经成为企业级 LLM 应用最常见的架构模式。"),
    Document(text="向量数据库是 RAG 架构中的关键组件。它将文本转换为高维向量后存储，支持基于语义相似度的快速检索。常见的向量数据库包括 FAISS、Chroma、Pinecone、Milvus 等。与传统数据库基于精确匹配不同，向量数据库支持模糊的语义搜索。"),
    Document(text="向量数据库和传统关系型数据库的区别在于：传统数据库使用 SQL 进行精确查询，适合结构化数据；向量数据库使用向量相似度进行语义检索，适合非结构化文本、图像等数据。两者在实际项目中经常互补使用。"),
]

index = VectorStoreIndex.from_documents(rag_docs)
print(f"索引构建完成，共 {len(rag_docs)} 个文档")

## 1. 对比：无状态查询 vs 有状态对话

前几章一直在用 `QueryEngine`。它功能强大，但有一个致命局限——**无状态**。每次查询都是独立的，没有对话记忆。

In [ ]:
# ❌ 查询引擎是无状态的——没有对话记忆
query_engine = index.as_query_engine()
print("问题1: 什么是 RAG？")
print(query_engine.query("什么是 RAG？"))
print()
print("问题2: 它有什么优势？")
print(query_engine.query("它有什么优势？"))  # "它"指代不明，查询引擎不知道上文

### 刚才发生了什么？

第二个问题「它有什么优势？」中的「它」指的是 RAG，但 `QueryEngine` 完全不知道之前聊过什么。它会把「它有什么优势？」当作一个独立的问题去检索，大概率检索不到有用的内容。

这就是为什么我们需要 **ChatEngine（对话引擎）**。

> **LangChain 对比**：LangChain 的链（Chain）也是无状态的，需要通过 `RunnableWithMessageHistory` 手动添加记忆。LlamaIndex 的情况类似——`QueryEngine` 无状态，`ChatEngine` 有状态。

In [ ]:
# ✅ 对话引擎有记忆——理解代词指代
chat_engine = index.as_chat_engine(chat_mode="condense_plus_context")
print("问题1: 什么是 RAG？")
print(chat_engine.chat("什么是 RAG？"))
print()
print("问题2: 它有什么优势？")
print(chat_engine.chat("它有什么优势？"))  # 理解"它"指 RAG

### 刚才发生了什么？

`ChatEngine` 维护了对话历史。当你问「它有什么优势？」时，它会结合上下文理解「它」指的是 RAG，然后去检索 RAG 的优势相关内容。

核心区别：

| | QueryEngine | ChatEngine |
|--|--|--|
| **状态** | 无状态 | 有状态（维护对话历史） |
| **创建方式** | `index.as_query_engine()` | `index.as_chat_engine()` |
| **调用方法** | `.query(str)` | `.chat(str)` |
| **适合场景** | 单轮问答、API 接口 | 多轮对话、聊天机器人 |

## 2. 对话模式详解

LlamaIndex 提供了三种对话模式，每种模式处理「后续问题」的方式不同。

| 模式 | 工作方式 | 适合场景 |
|------|---------|----------|
| `condense_question` | 将后续问题改写为独立问题，再查询索引 | 简单多轮问答 |
| `context` | 每次都检索上下文，拼接聊天历史一起发给 LLM | 需要持续参考知识库 |
| `condense_plus_context` | 先改写问题，再检索并拼接历史（两者结合） | 最全面，推荐默认使用 |

下面逐一演示。

### 2.1 condense_question 模式

工作流程：
1. 拿到用户的后续问题 + 聊天历史
2. 用 LLM 将后续问题改写为一个**独立的**、不依赖上下文的新问题
3. 用改写后的问题去检索索引
4. 根据检索结果生成回答

In [ ]:
# condense_question 模式
chat_engine_cq = index.as_chat_engine(chat_mode="condense_question")
print("【condense_question 模式】")
print("问: 什么是向量数据库？")
print(chat_engine_cq.chat("什么是向量数据库？"))
print()
print("问: 它和传统数据库有什么区别？")
print(chat_engine_cq.chat("它和传统数据库有什么区别？"))
chat_engine_cq.reset()  # 用完记得重置

### 刚才发生了什么？

`condense_question` 模式的核心操作是**问题改写**。当你问「它和传统数据库有什么区别？」时，它先把这个问题改写为「向量数据库和传统数据库有什么区别？」，再去检索。

优点：改写后的问题质量高，检索精准。
缺点：需要两次 LLM 调用（一次改写 + 一次回答），成本稍高。

### 2.2 context 模式

工作流程：
1. 直接用用户原始问题检索索引
2. 将检索到的上下文 + 聊天历史 + 用户问题一起发给 LLM
3. LLM 根据所有信息生成回答

In [ ]:
# context 模式
chat_engine_ctx = index.as_chat_engine(chat_mode="context")
print("【context 模式】")
print("问: 什么是向量数据库？")
print(chat_engine_ctx.chat("什么是向量数据库？"))
print()
print("问: 它和传统数据库有什么区别？")
print(chat_engine_ctx.chat("它和传统数据库有什么区别？"))
chat_engine_ctx.reset()

### 刚才发生了什么？

`context` 模式不改写问题，而是把聊天历史直接塞给 LLM。LLM 自己去理解「它」指代什么。

优点：只需要一次 LLM 调用，速度快、成本低。
缺点：用原始问题检索，「它和传统数据库有什么区别？」可能检索效果不如改写后的问题。

### 2.3 condense_plus_context 模式（推荐）

工作流程：
1. 先用 LLM 改写问题（和 condense_question 一样）
2. 用改写后的问题检索索引
3. 将检索结果 + 聊天历史 + 问题一起发给 LLM（和 context 一样）

综合了两种模式的优点，是最全面的模式。

In [ ]:
# condense_plus_context 模式（推荐）
chat_engine_cpc = index.as_chat_engine(chat_mode="condense_plus_context")
print("【condense_plus_context 模式】")
print("问: 什么是向量数据库？")
print(chat_engine_cpc.chat("什么是向量数据库？"))
print()
print("问: 它和传统数据库有什么区别？")
print(chat_engine_cpc.chat("它和传统数据库有什么区别？"))
chat_engine_cpc.reset()

### 刚才发生了什么？

`condense_plus_context` 模式是前两种模式的结合：先改写问题以提高检索质量，再将完整的聊天历史交给 LLM 以保持上下文连贯。

代价是需要**两次 LLM 调用**（改写 + 生成），但在大多数场景下效果最好。

### 模式选择建议

| 场景 | 推荐模式 | 原因 |
|------|---------|------|
| 快速原型 / 成本敏感 | `context` | 只需一次 LLM 调用 |
| 多轮问答质量优先 | `condense_plus_context` | 检索和生成都有优化 |
| 问题改写是主要瓶颈 | `condense_question` | 聚焦于改写质量 |

### 常见问题

- **三种模式回答差不多？** 对于简单问题确实如此。差异在复杂多轮对话中更明显，尤其是含有大量代词指代的场景。
- **`condense_plus_context` 比 `condense_question` 慢？** 两者都需要两次 LLM 调用，速度差异不大。区别在于 `condense_plus_context` 在最终回答时还会拼接聊天历史，生成效果更好。
- **如何查看改写后的问题？** 可以通过设置日志级别为 DEBUG 来查看中间步骤：`import logging; logging.basicConfig(level=logging.DEBUG)`。

## 3. 对话历史管理

ChatEngine 自动维护对话历史。我们可以查看、利用或重置它。

In [ ]:
# 创建对话引擎并进行几轮对话
chat_engine = index.as_chat_engine(chat_mode="condense_plus_context")
chat_engine.chat("什么是 RAG？")
chat_engine.chat("它的核心步骤是什么？")

# 查看对话历史
print("对话历史:")
for msg in chat_engine.chat_history:
    print(f"  [{msg.role.value}]: {str(msg.content)[:80]}...")

In [ ]:
# 重置对话——清空所有历史
chat_engine.reset()
print(f"重置后历史长度: {len(chat_engine.chat_history)}")

### 刚才发生了什么？

- **`chat_engine.chat_history`** 是一个列表，存储了所有对话消息（`ChatMessage` 对象），每条消息有 `role`（USER / ASSISTANT）和 `content`。
- **`chat_engine.reset()`** 清空所有对话历史，相当于开始一段全新的对话。

### 与 LangChain 的对比

| 功能 | LangChain | LlamaIndex |
|------|-----------|------------|
| 添加记忆 | `RunnableWithMessageHistory` 包装 + `ChatMessageHistory` | `as_chat_engine()` 内置 |
| 查看历史 | 从 `ChatMessageHistory` 对象获取 | `chat_engine.chat_history` |
| 重置对话 | 手动调用 `history.clear()` | `chat_engine.reset()` |
| 会话隔离 | 需要 `session_id` + `get_session_history` 函数 | 每个 ChatEngine 实例独立 |

LangChain 的记忆管理更灵活（支持多种存储后端、会话隔离），但配置也更复杂。LlamaIndex 的 ChatEngine 开箱即用，适合快速构建对话系统。

## 4. 多知识库场景

到目前为止，我们只用了一个索引。但实际项目中，你可能有多个知识库：产品文档、技术文档、FAQ、政策法规……

用户提问时，如何自动选择正确的知识库？

In [ ]:
# 创建两个独立的知识库
python_docs = [
    Document(text="Python 是一种解释型、面向对象的高级编程语言，由 Guido van Rossum 于 1991 年创建。Python 的设计哲学强调代码的可读性，使用显著的缩进来表示代码块。"),
    Document(text="Python 的列表推导式（list comprehension）是一种简洁创建列表的方式。语法为 [表达式 for 变量 in 可迭代对象 if 条件]。例如 [x**2 for x in range(10)] 生成 0 到 9 的平方数列表。"),
    Document(text="Python 的装饰器（decorator）是一种设计模式，使用 @decorator 语法糖。本质上是一个接受函数作为参数并返回新函数的高阶函数。常用于日志记录、权限验证、性能计时等横切关注点。"),
    Document(text="Python 的虚拟环境（virtual environment）用于隔离项目依赖。常用工具包括 venv（标准库）、virtualenv、conda 和 uv。每个虚拟环境有独立的 site-packages 目录。"),
]

javascript_docs = [
    Document(text="JavaScript 是一种动态类型的脚本语言，最初为浏览器端开发设计。现在通过 Node.js 也广泛用于服务器端开发。JavaScript 是 Web 开发的三大核心技术之一（HTML、CSS、JavaScript）。"),
    Document(text="JavaScript 的 Promise 是处理异步操作的核心机制。Promise 有三种状态：pending（进行中）、fulfilled（已完成）、rejected（已拒绝）。通过 .then() 和 .catch() 链式调用处理异步结果。"),
    Document(text="JavaScript 的箭头函数（arrow function）使用 => 语法，是 ES6 引入的简写形式。箭头函数没有自己的 this 绑定，它会捕获外层作用域的 this 值。语法：(参数) => 表达式 或 (参数) => { 语句块 }。"),
    Document(text="JavaScript 的 async/await 是基于 Promise 的语法糖，让异步代码看起来像同步代码。async 函数总是返回一个 Promise，await 只能在 async 函数内部使用。它使异步流程控制更加直观易读。"),
]

python_index = VectorStoreIndex.from_documents(python_docs)
js_index = VectorStoreIndex.from_documents(javascript_docs)

print(f"Python 知识库: {len(python_docs)} 个文档")
print(f"JavaScript 知识库: {len(javascript_docs)} 个文档")

### 刚才发生了什么？

我们创建了两个完全独立的 `VectorStoreIndex`，分别存储 Python 和 JavaScript 的知识。

如果只有一个 `QueryEngine`，面对「Python 的装饰器怎么用？」和「JavaScript 的 Promise 是什么？」这两个问题，需要在同一个索引中检索，可能会出现跨领域的噪音。

接下来我们要解决的问题是：**如何根据用户提问自动选择正确的知识库？**

## 5. QueryEngineTool：将索引包装为工具

要实现自动路由，第一步是把每个索引的查询引擎包装成「工具」，并为每个工具提供描述信息。路由器将根据这些描述来决定把问题发给哪个工具。

In [ ]:
from llama_index.core.tools import QueryEngineTool, ToolMetadata

python_tool = QueryEngineTool(
    query_engine=python_index.as_query_engine(),
    metadata=ToolMetadata(
        name="python_knowledge",
        description="Python 编程语言相关知识，包括语法、特性、装饰器、虚拟环境等",
    ),
)

js_tool = QueryEngineTool(
    query_engine=js_index.as_query_engine(),
    metadata=ToolMetadata(
        name="javascript_knowledge",
        description="JavaScript 编程语言相关知识，包括语法、异步编程、Promise、箭头函数等",
    ),
)

print(f"工具 1: {python_tool.metadata.name}")
print(f"  描述: {python_tool.metadata.description}")
print(f"工具 2: {js_tool.metadata.name}")
print(f"  描述: {js_tool.metadata.description}")

### 刚才发生了什么？

`QueryEngineTool` 将一个 `QueryEngine` 包装为「工具」对象，附带名称和描述。

- **`name`**：工具的唯一标识符
- **`description`**：关键！路由器/Agent 根据这个描述来判断该工具适合处理什么问题

> 描述写得好不好，直接影响路由的准确率。描述要具体、准确，避免模糊笼统。

### 与 LangChain 的对比

| | LangChain | LlamaIndex |
|--|-----------|------------|
| 工具定义 | `@tool` 装饰器 + 函数 | `QueryEngineTool` 专用类 |
| 描述方式 | 函数 docstring 或 `description` 参数 | `ToolMetadata(description=...)` |
| 集成程度 | 需要自己编写 retriever 调用逻辑 | 直接包装 QueryEngine，零额外代码 |

LangChain 中要实现类似功能，通常需要用 `@tool` 装饰一个自定义函数，函数内部调用 retriever。LlamaIndex 的 `QueryEngineTool` 省去了这层手动编写。

## 6. RouterQueryEngine：自动路由

有了工具，接下来用 `RouterQueryEngine` 实现自动路由。它的工作方式很直观：

1. 接收用户问题
2. 用 LLM 分析问题应该交给哪个工具
3. 调用对应工具的 QueryEngine
4. 返回结果

In [ ]:
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector

router_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=[python_tool, js_tool],
)

print("RouterQueryEngine 创建完成")
print(f"可用工具: {[t.metadata.name for t in router_engine._query_engine_tools]}")

In [ ]:
# 自动路由到正确的知识库
print("--- Python 问题 ---")
response = router_engine.query("Python 的装饰器怎么用？")
print(response)
print()

In [ ]:
print("--- JavaScript 问题 ---")
response = router_engine.query("JavaScript 的 Promise 是什么？")
print(response)

### 刚才发生了什么？

`RouterQueryEngine` 的核心是 **`LLMSingleSelector`**——它用 LLM 来分析用户问题，从所有工具的 `description` 中选择最匹配的那个。

- 问「Python 的装饰器」→ 匹配到 `python_knowledge` 工具
- 问「JavaScript 的 Promise」→ 匹配到 `javascript_knowledge` 工具

整个路由过程完全自动，不需要编写任何 if-else 规则。

> **与 LangChain 的对比**：在 LangChain 中，要实现相同的自动路由功能，通常需要构建一个 Agent（使用 `create_tool_calling_agent` 或类似方式），将 retriever 作为工具注册。LlamaIndex 的 `RouterQueryEngine` 提供了更轻量级的方案——不需要完整的 Agent 框架，只需要一个路由器。

### LLMSingleSelector vs LLMMultiSelector

LlamaIndex 提供了两种选择器：

| 选择器 | 行为 | 适用场景 |
|--------|------|----------|
| `LLMSingleSelector` | 选择**一个**最相关的工具 | 问题明确属于某一领域 |
| `LLMMultiSelector` | 可以选择**多个**工具 | 跨领域问题，需要综合多个知识库 |

大多数场景用 `LLMSingleSelector` 就够了。

## 7. 流式对话

ChatEngine 也支持流式输出，适合在 Web 应用中实现逐字显示的效果。

In [ ]:
# 流式对话
chat_engine = index.as_chat_engine(chat_mode="condense_plus_context")
response = chat_engine.stream_chat("详细解释 RAG 的工作流程")

for token in response.response_gen:
    print(token, end="", flush=True)
print()

### 刚才发生了什么？

- **`.stream_chat()`** 返回一个 `StreamingAgentChatResponse` 对象
- **`.response_gen`** 是一个生成器，逐步产出文本片段
- 使用 `end=""` 和 `flush=True` 让输出逐字显示

与前面章节学过的 `query_engine.query()` 的流式版本类似，区别在于 `stream_chat()` 会维护对话历史。

| 方法 | 返回类型 | 是否维护历史 |
|------|---------|-------------|
| `query_engine.query()` | `Response` | 否 |
| `chat_engine.chat()` | `AgentChatResponse` | 是 |
| `chat_engine.stream_chat()` | `StreamingAgentChatResponse` | 是 |

In [ ]:
# 流式对话同样维护历史
response2 = chat_engine.stream_chat("请用三个要点总结")
for token in response2.response_gen:
    print(token, end="", flush=True)
print()

print(f"\n对话历史长度: {len(chat_engine.chat_history)}")
chat_engine.reset()

## 8. 综合对比：LangChain vs LlamaIndex

本章涉及的功能，两个框架的实现方式对比：

| 功能 | LangChain | LlamaIndex |
|------|-----------|------------|
| 对话记忆 | `RunnableWithMessageHistory` + 手动管理 session | `as_chat_engine()` 内置记忆 |
| 会话重置 | 手动调用 `history.clear()` | `chat_engine.reset()` |
| 对话模式 | 需自行设计 chain 逻辑 | `chat_mode` 参数一键切换 |
| 工具包装 | `@tool` 装饰器 + 自定义函数 | `QueryEngineTool` 专用封装 |
| 自动路由 | 需要构建完整 Agent（`create_tool_calling_agent`） | `RouterQueryEngine` 开箱即用 |
| 流式对话 | `.stream()` 方法 + 回调 | `stream_chat()` + `response_gen` |

**一句话总结**：LlamaIndex 在 RAG 对话和多知识库路由上提供了更多开箱即用的方案；LangChain 则更灵活，适合需要高度自定义的场景。两者不是替代关系，而是互补。

## 总结

本章学习了：
- QueryEngine（无状态）vs ChatEngine（有状态）的核心区别
- 三种对话模式：`condense_question` / `context` / `condense_plus_context`
- 对话历史的查看（`chat_history`）和重置（`reset()`）
- `QueryEngineTool` 将索引包装为可路由的工具
- `RouterQueryEngine` + `LLMSingleSelector` 实现自动路由
- 流式对话 `stream_chat()` 的用法

## 下一章预告

**第七章：生产实践与完整项目** —— 最后一章将学习 RAG 评估指标、可观测性调试，并构建一个端到端的多源问答系统，整合前六章所有知识。